# Exercises XP – Week 5, Day 5: Introduction to Deep Learning
Runs best in **Google Colab** (TensorFlow is pre-installed there). Locally: `pip install tensorflow`.

## 🌟 Exercise 1: Small Quiz

**1. What is the key difference between traditional machine learning and deep learning?**
In traditional ML, a person usually has to design the input features by hand (e.g. "loan-to-income ratio"), and the model learns from those features. In deep learning, a neural network with many layers **learns the features itself** directly from raw data (pixels, text, sound). Deep learning also needs much more data and computing power.

**2. How do artificial neural networks (ANNs) mimic the human brain?**
They are made of many simple units called **neurons**, connected to each other like brain neurons are connected by synapses. Each artificial neuron receives inputs, multiplies them by **weights** (like the strength of a synapse), adds them up, and "fires" through an activation function. Learning means adjusting those weights based on experience (training data). The analogy is loose – real brains are far more complex – but the idea of many connected units learning by strengthening or weakening connections is the same.

**3. Why does deep learning perform better on large datasets compared to traditional machine learning?**
Deep networks have a huge number of parameters, so they can represent very complex patterns. With little data they overfit, but with a lot of data they can learn those complex patterns reliably, and their performance keeps improving as more data is added. Traditional models usually reach a plateau: after a certain amount of data, adding more doesn't help much.

**4. What are some challenges of deep learning, and how can they be addressed?**
- *Needs a lot of data* → data augmentation, transfer learning (starting from a pre-trained model).
- *Overfitting* → dropout, regularisation, early stopping, more data.
- *High computing cost* → GPUs/TPUs, cloud computing, smaller or pre-trained models.
- *"Black box" – hard to interpret* → explainability tools (e.g. SHAP, saliency maps).
- *Many hyperparameters to tune* → systematic search (grid/random search), sensible defaults.

**5. What is feature engineering, and why is it not needed in deep learning?**
Feature engineering means creating or transforming input variables by hand, using domain knowledge, so that a model can learn more easily (for example, extracting edges from an image or computing ratios). In deep learning it is **largely not needed** because the hidden layers learn useful representations automatically: in an image network, early layers learn edges, later ones learn shapes and objects. (Some data cleaning and preparation is still needed.)

**6. What role do hidden layers play in a deep learning model?**
Hidden layers sit between the input and the output and **transform the data step by step** into more abstract, useful representations. Each layer combines the outputs of the previous one, so the network can learn more and more complex patterns. More hidden layers ("deeper" networks) can model more complex relationships.

**7. In an ANN, what is the function of an activation function?**
It decides each neuron's output and, most importantly, adds **non-linearity**. Without activation functions, any number of layers would collapse into one simple linear function, and the network could only learn straight-line relationships. Common ones are **ReLU** (hidden layers), **sigmoid** (binary output) and **softmax** (probabilities across several classes).

## 🌟 Exercise 2: Building a Simple Perceptron Decision System

In [ ]:
def perceptron(temperature, rain, w_temp=0.6, w_rain=0.4, bias=2):
    weighted_sum = temperature * w_temp + rain * w_rain + bias
    output = 1 if weighted_sum > 20 else 0      # step activation function
    return weighted_sum, output

for case, (temp, rain) in {"Case 1": (70, 0), "Case 2": (50, 1)}.items():
    ws, out = perceptron(temp, rain)
    decision = "Yes, go outside" if out == 1 else "No, stay inside"
    print(f"{case}: Temperature={temp}°F, Rain={rain} → weighted sum = {ws:.1f} → output {out} ({decision})")

**Interpretation:** the perceptron suggested going outside **in both cases** (weighted sums 44 and 32.4, both above 20). The temperature weight is large, so temperature dominates the decision, and the rain weight is **positive** (+0.4), so rain actually *increases* the sum slightly instead of discouraging us. To make rain a reason to stay inside, its weight would need to be **negative** (e.g. −15). This shows that a perceptron's "decisions" depend entirely on the weights – which is exactly what training learns automatically.

## 🌟 Exercise 3: Building a Simple Neural Network with TensorFlow/Keras (MNIST)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# 2. Load the data
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print("Train:", x_train.shape, "Test:", x_test.shape)

# 3. Normalize pixel values to [0, 1]
x_train = x_train / 255.0
x_test = x_test / 255.0

# 4. One-hot encode the labels (e.g. 3 → [0,0,0,1,0,0,0,0,0,0])
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

In [ ]:
# 5. Build the model
model = models.Sequential([
    layers.Input(shape=(28, 28)),             # input layer
    layers.Flatten(),                         # 28x28 image → 784 numbers
    layers.Dense(128, activation="relu"),     # hidden layer
    layers.Dense(10, activation="softmax"),   # output: one probability per digit
])

# 6. Compile
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# 7. Train and evaluate
history = model.fit(x_train, y_train_cat, epochs=5, batch_size=32, validation_split=0.1)

test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

With one hidden layer of 128 neurons and 5 epochs, this network typically reaches about **97–98% accuracy** on the test images – very good for such a simple model.

## 🌟 Exercise 4: Forward Propagation Calculation

In [ ]:
x1, x2 = 2000, 3          # square footage, bedrooms
w1, w2 = 0.5, 0.7         # weights
b = 50_000                # bias

z = x1 * w1 + x2 * w2 + b            # weighted sum before activation
prediction = max(0, z)               # ReLU(z) = max(0, z)

print("z = (2000 × 0.5) + (3 × 0.7) + 50,000 =", z)
print("ReLU(z) =", prediction)

- **z** = 1,000 + 2.1 + 50,000 = **51,002.1**
- **ReLU** keeps positive values unchanged, so the prediction is **51,002.1**.
- **Interpretation:** the network predicts a house price of about **51,002 currency units**. Almost all of it comes from the bias (50,000); the square footage adds 1,000 and the bedrooms add only 2.1. With these untrained weights the prediction isn't realistic – training would adjust the weights so that the features matter much more.

## 🌟 Exercise 5 (optional): Implementing Forward and Backward Propagation in Python

In [ ]:
import numpy as np

x = np.array([4, 80])        # 4 hours studied, previous test score: 80
w = np.array([0.6, 0.3])     # initial weights
b = 10                       # initial bias

def forward_propagation(x, w, b):
    z = np.dot(x, w) + b     # weighted sum
    return z                 # linear activation (regression)

y_pred = forward_propagation(x, w, b)
y_true = 85

loss = 0.5 * (y_true - y_pred) ** 2

grad_w = -(y_true - y_pred) * x
grad_b = -(y_true - y_pred)

learning_rate = 0.01
w_new = w - learning_rate * grad_w
b_new = b - learning_rate * grad_b

print("Initial Prediction:", y_pred)
print("Loss:", loss)
print("Updated Weights:", w_new)
print("Updated Bias:", b_new)

In [ ]:
# Train for several steps to see the loss go down, and compare learning rates
def train(w, b, learning_rate, steps=10):
    losses = []
    for _ in range(steps):
        y_pred = forward_propagation(x, w, b)
        losses.append(0.5 * (y_true - y_pred) ** 2)
        error = y_true - y_pred
        w = w + learning_rate * error * x
        b = b + learning_rate * error
    return losses

for lr in [0.0001, 0.0002, 0.0003]:
    losses = train(np.array([0.6, 0.3]), 10, lr)
    print(f"learning rate {lr}: loss after 1 step = {losses[1]:.2f}, after 10 steps = {losses[-1]:.4f}")

print("\nWith the original learning rate 0.01:")
print([f"{l:,.0f}" for l in train(np.array([0.6, 0.3]), 10, 0.01, steps=4)])

**Why gradient descent reduces the error:** the gradient tells us in which direction (and how strongly) each weight increases the loss. Moving the weights a small step in the **opposite** direction of the gradient lowers the loss. Here the first prediction (36.4) is far below the true score (85), so the gradients are negative and the update **increases** the weights and bias, pushing the next prediction up toward 85.

**Effect of the learning rate:** with small learning rates (0.0001 and 0.0002) the loss drops to almost 0 within 10 steps. At 0.0003 the steps are already a bit too big: the prediction jumps past 85 and back, and the loss is still high after 10 steps. With **0.01 the step is far too big** for these inputs (the "80" feature is large), so the prediction overshoots 85 by a lot and the loss **explodes** instead of shrinking. This is why choosing the learning rate (and scaling the inputs) matters so much.

## 🌟 Exercise 6 (optional): Visualizing Predictions on the MNIST Dataset

In [ ]:
# Re-uses the trained model from Exercise 3
predictions = model.predict(x_test[:15], verbose=0)
predicted_labels = np.argmax(predictions, axis=1)

plt.figure(figsize=(12, 5))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[i], cmap="gray")
    color = "green" if predicted_labels[i] == y_test[i] else "red"
    plt.title(f"Pred: {predicted_labels[i]} (true {y_test[i]})", color=color, fontsize=9)
    plt.axis("off")
plt.suptitle("MNIST test images with predicted labels (red = wrong)")
plt.tight_layout()
plt.show()